In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv('../results/HELMSamplesLabeled.csv')

metrics = ['stats_bleu_4','stats_f1_score','Vee','Eee','Ven','Een']

In [ ]:
for col in metrics:
    vals = df[col].dropna()
    y = df.loc[vals.index,'label']
    auc = roc_auc_score(y, vals)
    sns.histplot(vals[y==1], color='green', label='correct', kde=True)
    sns.histplot(vals[y==0], color='red', label='incorrect', kde=True)
    plt.title(f"{col} (AUC={auc:.3f})")
    plt.legend(); plt.show()

In [ ]:
for m in metrics:
    if m in df:
        corr = df[[m,'label']].corr(method='spearman').iloc[0,1]
        print(f"{m:15s}  ρ={corr:.3f}")

In [ ]:
results = []

for m in metrics:
    if m not in df.columns:
        continue
    
    tmp = df[['label', m]].dropna()
    if tmp[m].nunique() <= 1:
        continue  

    y = tmp['label']
    x = tmp[m].astype(float)

    auc = roc_auc_score(y, x)
    ap  = average_precision_score(y, x)

    results.append((m, auc, ap))
    print(f"{m:15s}  ROC-AUC={auc:.3f}  PR-AUC={ap:.3f}")

In [ ]:
for m in ['Vee','Eee','Ven','Een']:
    if m in df:
        tmp = df[['label', m]].dropna()
        acc = (tmp['label'] == tmp[m]).mean()
        print(f"{m:15s}  accuracy={acc:.3f}")

In [ ]:
SEED = 42

f1 = df[df['stats_f1_score'].notna()]
min_count = f1['label'].value_counts().min()
f1_balanced = (f1.groupby('label', group_keys=False).apply(lambda x: x.sample(min_count, random_state=SEED)).reset_index(drop=True))
scores = f1_balanced["stats_f1_score"].values
labels = f1_balanced["label"].values
print(f"length: {len(f1_balanced)}")

thresholds = np.sort(np.unique(scores))
accuracies = [(labels == (scores >= t)).mean() for t in thresholds]

best_idx = np.argmax(accuracies)
best_threshold = thresholds[best_idx]
best_accuracy = accuracies[best_idx]

print(f"Best f1 threshold (balanced) = {best_threshold:.4f}, Accuracy = {best_accuracy:.4f}")

plt.figure(figsize=(7,4))
plt.plot(thresholds, accuracies, label="Accuracy", color="blue")
plt.axvline(best_threshold, color="red", linestyle="--", label=f"Best threshold = {best_threshold:.3f}")
plt.scatter([best_threshold], [best_accuracy], color="red", zorder=5)
plt.title("Threshold vs Accuracy")
plt.xlabel("Threshold")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.xlim(0, 1)
plt.show()

bleu = df[df['stats_bleu_4'].notna()]
min_count = bleu['label'].value_counts().min()
bleu_balanced = (bleu.groupby('label', group_keys=False).apply(lambda x: x.sample(min_count, random_state=SEED)).reset_index(drop=True))
scores = bleu_balanced["stats_bleu_4"].values
labels = bleu_balanced["label"].values
print(f"length: {len(bleu_balanced)}")

thresholds = np.sort(np.unique(scores))
accuracies = [(labels == (scores >= t)).mean() for t in thresholds]

best_idx = np.argmax(accuracies)
best_threshold = thresholds[best_idx]
best_accuracy = accuracies[best_idx]

print(f"Best bleu threshold (balanced) = {best_threshold:.4f}, Accuracy = {best_accuracy:.4f}")

plt.figure(figsize=(7,4))
plt.plot(thresholds, accuracies, label="Accuracy", color="blue")
plt.axvline(best_threshold, color="red", linestyle="--", label=f"Best threshold = {best_threshold:.3f}")
plt.scatter([best_threshold], [best_accuracy], color="red", zorder=5)
plt.title("Threshold vs Accuracy")
plt.xlabel("Threshold")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#optimal thresholds on unbalanced dataset
f1 = df[df['stats_f1_score'].notna()]
scores = f1["stats_f1_score"].values
labels = f1["label"].values

thresholds = np.sort(np.unique(scores))
accuracies = [(labels == (scores >= t)).mean() for t in thresholds]

best_idx = np.argmax(accuracies)
best_threshold = thresholds[best_idx]
best_accuracy = accuracies[best_idx]

print(f"Best f1 threshold = {best_threshold:.4f}, Accuracy = {best_accuracy:.4f}")

plt.figure(figsize=(7,4))
plt.plot(thresholds, accuracies, label="Accuracy", color="blue")
plt.axvline(best_threshold, color="red", linestyle="--", label=f"Best threshold = {best_threshold:.3f}")
plt.scatter([best_threshold], [best_accuracy], color="red", zorder=5)
plt.title("Threshold vs Accuracy")
plt.xlabel("Threshold")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

bleu = df[df['stats_bleu_4'].notna()]
scores = bleu["stats_bleu_4"].values
labels = bleu["label"].values

thresholds = np.sort(np.unique(scores))
accuracies = [(labels == (scores >= t)).mean() for t in thresholds]

best_idx = np.argmax(accuracies)
best_threshold = thresholds[best_idx]
best_accuracy = accuracies[best_idx]

print(f"Best bleu threshold = {best_threshold:.4f}, Accuracy = {best_accuracy:.4f}")

plt.figure(figsize=(7,4))
plt.plot(thresholds, accuracies, label="Accuracy", color="blue")
plt.axvline(best_threshold, color="red", linestyle="--", label=f"Best threshold = {best_threshold:.3f}")
plt.scatter([best_threshold], [best_accuracy], color="red", zorder=5)
plt.title("Threshold vs Accuracy")
plt.xlabel("Threshold")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()